In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_absolute_error
from statsmodels.tsa.seasonal import STL

In [2]:
df = pd.read_csv("/content/city_day.csv")

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['City','Date'])

print("Original Shape:", df.shape)

Original Shape: (29531, 16)


In [3]:
pollutants = [
'PM2.5','PM10','NO','NO2','NOx','NH3',
'CO','SO2','O3','Benzene','Toluene','Xylene','AQI'
]

In [4]:
def smart_fill(city_df):

    city_df = city_df.copy()

    # 1️⃣ Linear interpolation (time based)
    city_df[pollutants] = city_df[pollutants].interpolate(
        method='linear',
        limit_direction='both'
    )

    # 2️⃣ Rolling mean smoothing
    city_df[pollutants] = city_df[pollutants].fillna(
        city_df[pollutants].rolling(7, min_periods=1).mean()
    )

    # 3️⃣ Seasonal monthly mean
    city_df['month'] = city_df['Date'].dt.month

    for col in pollutants:
        city_df[col] = city_df[col].fillna(
            city_df.groupby('month')[col].transform('mean')
        )

    # remove helper column
    city_df.drop(columns=['month'], inplace=True)

    return city_df

df = df.groupby("City").apply(smart_fill)
df.reset_index(drop=True, inplace=True)

/tmp/ipython-input-639/3244481756.py:29: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("City").apply(smart_fill)


In [5]:
from sklearn.impute import KNNImputer

imputer = KNNImputer(n_neighbors=5)

df[pollutants] = imputer.fit_transform(df[pollutants])

In [6]:
print("Remaining Missing Values:")
print(df[pollutants].isnull().sum())

Remaining Missing Values:
PM2.5      0
PM10       0
NO         0
NO2        0
NOx        0
NH3        0
CO         0
SO2        0
O3         0
Benzene    0
Toluene    0
Xylene     0
AQI        0
dtype: int64


In [7]:
df.shape

(29531, 16)

In [8]:
from statsmodels.tsa.seasonal import STL

def apply_stl(city_df):
    city_df = city_df.copy()

    stl = STL(city_df['AQI'], period=365, robust=True)
    result = stl.fit()

    city_df['trend'] = result.trend
    city_df['seasonal'] = result.seasonal
    city_df['residual'] = result.resid

    return city_df

df = df.groupby("City", group_keys=False).apply(apply_stl)

/tmp/ipython-input-639/1386118091.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("City", group_keys=False).apply(apply_stl)


In [9]:
print(df[['trend','seasonal','residual']].head())

        trend    seasonal   residual
0    9.630609  104.369391   0.000000
1  277.807570  -45.234517 -23.573053
2  277.931266  -35.328171 -33.603095
3  278.054888  -32.072517 -36.982371
4  278.178436  -39.211646 -29.966791


In [10]:
features = [
'PM2.5','PM10','NO','NO2','NOx','NH3',
'CO','SO2','O3','Benzene','Toluene','Xylene',
'trend','seasonal','residual'
]

target = 'AQI'

In [11]:
from sklearn.preprocessing import MinMaxScaler

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

df[features] = scaler_X.fit_transform(df[features])
df[[target]] = scaler_y.fit_transform(df[[target]])

In [12]:
import numpy as np

def create_sequences(data, targets, seq_len=30):
    X, y = [], []

    for i in range(len(data)-seq_len):
        X.append(data[i:i+seq_len])
        y.append(targets[i+seq_len])

    return np.array(X), np.array(y)

In [13]:
X_all, y_all = [], []

for city in df['City'].unique():

    city_df = df[df['City']==city]

    data = city_df[features].values
    targets = city_df[target].values

    if len(data) > 50:
        X,y = create_sequences(data, targets)
        X_all.append(X)
        y_all.append(y)

X_all = np.concatenate(X_all)
y_all = np.concatenate(y_all)

print("Final Dataset Shape:", X_all.shape)

Final Dataset Shape: (28750, 30, 15)


In [14]:
split = int(len(X_all)*0.8)

X_train = X_all[:split]
y_train = y_all[:split]

X_test = X_all[split:]
y_test = y_all[split:]

In [15]:
import torch

X_train = torch.FloatTensor(X_train)
y_train = torch.FloatTensor(y_train).view(-1,1)

X_test = torch.FloatTensor(X_test)
y_test = torch.FloatTensor(y_test).view(-1,1)

In [16]:
import torch.nn as nn

class AQIModel(nn.Module):
    def __init__(self,input_dim):
        super().__init__()

        self.embedding = nn.Linear(input_dim,64)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=64,
            nhead=4,
            dropout=0.1,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        self.fc = nn.Linear(64,1)

    def forward(self,x):
        x = self.embedding(x)
        x = self.transformer(x)
        x = x[:,-1,:]
        return self.fc(x)

model = AQIModel(len(features))

In [17]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,   # Safe batch size
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False
)

In [18]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
loss_fn = nn.MSELoss()

In [25]:
# ===============================
# FAST TRAINING FIX (ALL 5 STEPS)
# ===============================

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# ---------- STEP 1 : CPU OPTIMIZATION ----------
torch.set_num_threads(4)   # change based on CPU cores

# ---------- STEP 2 : FAST MODEL (GRU instead of Transformer) ----------
class FastAQIModel(nn.Module):

    def __init__(self, input_dim):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=32,
            num_layers=1,
            batch_first=True
        )

        self.fc = nn.Linear(32,1)

    def forward(self,x):
        _, h = self.gru(x)
        return self.fc(h[-1])

model = FastAQIModel(X_train.shape[2])

# ---------- STEP 3 : LARGE BATCH TRAINING ----------
train_dataset = TensorDataset(X_train, y_train)

train_loader = DataLoader(
    train_dataset,
    batch_size=512,     # FAST TRAINING
    shuffle=True
)

# ---------- STEP 4 : OPTIMIZER ----------
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()



In [26]:
epochs = 25

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for batch_X, batch_y in train_loader:

        optimizer.zero_grad()

        preds = model(batch_X)
        loss = loss_fn(preds, batch_y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}")

Epoch 1, Loss: 0.006504761316399607
Epoch 2, Loss: 0.0025740084160740177
Epoch 3, Loss: 0.0021335316511491933
Epoch 4, Loss: 0.0019568985095247625
Epoch 5, Loss: 0.0018583670491352678
Epoch 6, Loss: 0.001783611382254296
Epoch 7, Loss: 0.001721579699207925
Epoch 8, Loss: 0.0016639937323311136
Epoch 9, Loss: 0.001608115143608302
Epoch 10, Loss: 0.001545521531564494
Epoch 11, Loss: 0.00146814515400264
Epoch 12, Loss: 0.001383066375274211
Epoch 13, Loss: 0.0012759951706458298
Epoch 14, Loss: 0.0011810723873269227
Epoch 15, Loss: 0.0010936232283711433
Epoch 16, Loss: 0.0010865946287392742
Epoch 17, Loss: 0.0010155150518080013
Epoch 18, Loss: 0.0009982309621086138
Epoch 19, Loss: 0.0009835189790464937
Epoch 20, Loss: 0.0009513669763691723
Epoch 21, Loss: 0.0009529139652537803
Epoch 22, Loss: 0.0009336689318944183
Epoch 23, Loss: 0.0009368099518016809
Epoch 24, Loss: 0.0009099340092183815
Epoch 25, Loss: 0.0009129793447856274


In [31]:
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    explained_variance_score,
    max_error
)
import numpy as np

# ======================
# Model Prediction
# ======================

model.eval()

with torch.no_grad():
    preds = model(X_test)

preds = preds.cpu().numpy()
y_true = y_test.cpu().numpy()

# ======================
# Metrics Calculation
# ======================

r2 = r2_score(y_true, preds)
mae = mean_absolute_error(y_true, preds)
mse = mean_squared_error(y_true, preds)
rmse = np.sqrt(mse)
mape = mean_absolute_percentage_error(y_true, preds)
evs = explained_variance_score(y_true, preds)
maxerr = max_error(y_true, preds)

# ======================
# Print Results
# ======================

print("\n===== MODEL EVALUATION METRICS =====")

print(f"R2 Score (Accuracy)      : {r2:.4f}")
print(f"MAE (Mean Abs Error)     : {mae:.4f}")
print(f"MSE (Mean Sq Error)      : {mse:.4f}")
print(f"RMSE                     : {rmse:.4f}")
print(f"MAPE (%)                 : {mape*100:.2f}")
print(f"Explained Variance Score : {evs:.4f}")
print(f"Max Error                : {maxerr:.4f}")


===== MODEL EVALUATION METRICS =====
R2 Score (Accuracy)      : 0.9028
MAE (Mean Abs Error)     : 0.0110
MSE (Mean Sq Error)      : 0.0003
RMSE                     : 0.0159
MAPE (%)                 : 1557441536000.00
Explained Variance Score : 0.9029
Max Error                : 0.1376


In [32]:
preds_real = scaler_y.inverse_transform(preds)
y_real = scaler_y.inverse_transform(y_true)

print("Real R2:", r2_score(y_real, preds_real))
print("Real MAE:", mean_absolute_error(y_real, preds_real))

Real R2: 0.9028446078300476
Real MAE: 22.367307662963867
